# AgriAgent: fine-tune Qwen2.5 with LoRA for the pipeline's three questions

This notebook trains a small adapter (LoRA) on top of **Qwen/Qwen2.5-1.5B-Instruct** so that the model answers the three questions the AgriAgent pipeline asks it:

| Task | The model reads | The model answers |
|---|---|---|
| `map_columns` | a table title and its column headers (Hindi or English, sometimes damaged) | crop, measure and unit for each column; what the first column holds |
| `route` | a short profile of the input file | which agent should take it |
| `doc_context` | the first lines of a document | district and year, or null |

It then scores the **base model and the adapter on the same held-out cases** and writes `eval/adapter_comparison.csv` (protocol item 10, metric M18).

**What you need:** a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU) and the project zip. Expect very roughly 45 to 90 minutes in total on a T4; this estimate has not been measured.

**Read before you report the result**
- The numbers in the database never come from the model. The adapter only improves labels (which crop, which unit).
- Most training examples are synthetic tables with known answers; the real-table examples are labelled by the project's own rules and are not human-checked.
- If `finetune/split_manifest.json` lists no held-out real files, the comparison is on synthetic held-out cases only. Say so. To hold real files out, put other districts' PDFs in `data/raw/` and re-run `python finetune/build_dataset.py` before this notebook.
- M18 rewards an honest, explained comparison. A gain of zero, or a loss, is a valid result to report.

In [ ]:
# 1. Packages (Colab already has torch). Safe to re-run.
import importlib, importlib.util, importlib.metadata, subprocess, sys
need = [p for p in ("transformers", "peft", "accelerate") if importlib.util.find_spec(p) is None]
if need: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.44", "peft>=0.12", "accelerate"])
# Colab ships an old torchao (0.10). Recent peft refuses to build a LoRA model when it finds a torchao older than 0.16.
# This notebook does not use torchao, so remove it. If peft was already imported in this session, restart the session after this cell.
try:
    tv = importlib.metadata.version("torchao")
    if tuple(int(x) for x in tv.split(".")[:2]) < (0, 16):
        subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"]); importlib.invalidate_caches()
        print(f"removed torchao {tv} (not used here; it blocks peft)")
        if "peft" in sys.modules: print(">>> peft was already imported: Runtime -> Restart session, then run all cells again.")
except importlib.metadata.PackageNotFoundError:
    pass
import torch, transformers, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")


## 2. Find the project

Run the next cell. If the notebook is not sitting inside the project folder, it asks you to upload the project zip (Colab) and unpacks it.

In [ ]:
import glob, os, sys, zipfile
PROJECT = os.environ.get("AGRI_PROJECT", "")
if not PROJECT:
    here = [p for p in (".", "..", "agriagent", "/content/agriagent") if os.path.exists(os.path.join(p, "finetune", "ft_utils.py"))]
    if here: PROJECT = os.path.abspath(here[0])
    else:
        from google.colab import files                      # upload agriagent_final.zip
        up = files.upload(); zipfile.ZipFile(next(iter(up))).extractall("/content")
        PROJECT = glob.glob("/content/**/finetune/ft_utils.py", recursive=True)[0].rsplit(os.sep + "finetune" + os.sep, 1)[0]
sys.path[:0] = [PROJECT, os.path.join(PROJECT, "finetune")]
import ft_utils as ft
print("project:", PROJECT)

In [ ]:
# 3. Settings. Change these, not the code below.
BASE_MODEL   = "Qwen/Qwen2.5-1.5B-Instruct"     # Apache-2.0. "Qwen/Qwen2.5-0.5B-Instruct" is faster and weaker.
ADAPTER_DIR  = os.path.join(PROJECT, "finetune", "adapter_qwen25_agri")
EPOCHS, LR   = 3, 2e-4
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
MAX_LEN      = 1536                              # tokens per training example (longer ones are cut)
BATCH, ACCUM = 1, 16                             # 1 x 16 = 16 examples per optimiser step. Out of GPU memory (CUBLAS_STATUS_ALLOC_FAILED)? keep BATCH 1, lower MAX_LEN or use the 0.5B model
EVAL_CASES   = 120                               # held-out cases scored for base and adapter (None = all)
SMOKE        = os.environ.get("AGRI_FT_SMOKE") == "1"   # code check with a tiny random model; its scores mean nothing
SEED         = 7

In [ ]:
# 4. Data, and the two overlap checks M18 asks for (must both be 0).
import json, random
train   = ft.load_jsonl(os.path.join(PROJECT, "finetune", "train.jsonl"))
heldout = ft.load_jsonl(os.path.join(PROJECT, "finetune", "heldout.jsonl"))
manifest = json.load(open(os.path.join(PROJECT, "finetune", "split_manifest.json"), encoding="utf-8"))
assert manifest["overlap_files_by_sha256"] == 0 and manifest["overlap_examples_by_prompt_hash"] == 0, "train and held-out overlap: rebuild the dataset"
prompts = lambda rows: {r["task"] + r["messages"][1]["content"] for r in rows}
assert not (prompts(train) & prompts(heldout)), "a held-out prompt is also in train"
shas = lambda rows: {r["source_sha256"] for r in rows if r["source_sha256"]}
assert not (shas(train) & shas(heldout)), "a held-out file is also a training file"
random.Random(SEED).shuffle(heldout)
cases = heldout[:EVAL_CASES] if EVAL_CASES else heldout
if SMOKE: train, cases = train[:48], cases[:6]
print(len(train), "training examples |", len(cases), "held-out cases scored |", "held-out real files:", list(manifest["heldout_files"]) or "none (synthetic held-out only)")
print({k: v for k, v in manifest["train_counts"].items() if v})

In [ ]:
# 5. Base model and tokenizer.
from transformers import AutoModelForCausalLM, AutoTokenizer
if SMOKE:
    tok, model = ft.tiny_model([m["content"] for r in train + cases for m in r["messages"]]); BASE_MODEL = "tiny-random-qwen2 (smoke test)"
else:
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)          # float32 weights; mixed precision is used during training and generation
if tok.pad_token_id is None: tok.pad_token = tok.eos_token
model.to("cuda" if torch.cuda.is_available() else "cpu")
print(BASE_MODEL, "|", round(sum(p.numel() for p in model.parameters()) / 1e6), "M parameters |", model.device)

In [ ]:
# 6. Baseline: the base model on the held-out cases, before any training.
import time
def run_cases(m, label):
    out, t0 = {}, time.time(); m.eval()
    for i, c in enumerate(cases):
        n_ref = len(tok(c["messages"][2]["content"], add_special_tokens=False)["input_ids"])
        out[c["id"]] = ft.generate(m, tok, c["messages"], max_new_tokens=(8 if SMOKE else min(700, n_ref + 60)))
        if (i + 1) % 20 == 0: print(f"  {label}: {i + 1}/{len(cases)} cases, {time.time() - t0:.0f} s")
    return out
base_out = run_cases(model, "base")
print("example base answer:", base_out[cases[0]["id"]][:300])

In [ ]:
# 7. Train the LoRA adapter. Only the assistant's answer is trained on (prompt tokens are masked).
#    A plain PyTorch loop is used (no Trainer), so the cell does not depend on which transformers version Colab has.
import gc, math
from peft import LoraConfig, get_peft_model
gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None   # free memory left over from scoring the base model
enc = [ft.encode_example(tok, r["messages"], MAX_LEN) for r in train]
cut = sum(len(e["input_ids"]) >= MAX_LEN for e in enc)
print("tokens per example: median", sorted(len(e["input_ids"]) for e in enc)[len(enc) // 2], "| max", max(len(e["input_ids"]) for e in enc), "| cut at MAX_LEN:", cut)
model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, task_type="CAUSAL_LM",
                                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()
# gradient checkpointing: recompute activations in the backward pass instead of storing them (far less GPU memory, about 30% slower)
model.config.use_cache = False; model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False}); model.enable_input_require_grads()
cuda = model.device.type == "cuda"; accum = 1 if SMOKE else ACCUM
total_steps = 4 if SMOKE else math.ceil(len(enc) / (BATCH * accum)) * EPOCHS
warm = max(1, int(0.05 * total_steps))
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.0)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total_steps))))
scaler = torch.amp.GradScaler("cuda", enabled=cuda)
model.train(); step, seen, running, t0 = 0, 0, 0.0, time.time(); last_loss = float("nan")
for epoch in range(EPOCHS):
    order = list(range(len(enc))); random.Random(SEED + epoch).shuffle(order)
    for k in range(0, len(order), BATCH):
        batch = {n: v.to(model.device) for n, v in ft.collate([enc[j] for j in order[k:k + BATCH]], tok.pad_token_id).items()}
        with torch.autocast("cuda", dtype=torch.float16, enabled=cuda): loss = model(**batch).loss
        scaler.scale(loss / accum).backward(); running += loss.item(); seen += 1
        if seen % accum == 0:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); step += 1
            if step % (1 if SMOKE else 10) == 0 or step == total_steps:
                last_loss = running / accum; print(f"  step {step}/{total_steps} | loss {last_loss:.4f} | {(time.time() - t0) / 60:.1f} min")
            running = 0.0
        if step >= total_steps: break
    if step >= total_steps: break
train_minutes = (time.time() - t0) / 60; global_step = step
model.gradient_checkpointing_disable(); model.config.use_cache = True; model.eval()      # back to fast generation for scoring
if cuda: print(f"peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")
print(f"trained {global_step} steps in {train_minutes:.1f} min | last logged training loss {last_loss:.4f}")

In [ ]:
# 8. The adapter on the SAME held-out cases.
adapter_out = run_cases(model, "adapter")
print("example adapter answer:", adapter_out[cases[0]["id"]][:300])

In [ ]:
# 9. Compare, and write eval/adapter_comparison.csv (+ eval/results/adapter_cases.csv, one row per case).
summary, per_case = ft.compare(cases, base_out, adapter_out)
info = f"LoRA r={LORA_R} alpha={LORA_ALPHA} dropout={LORA_DROPOUT}; epochs={EPOCHS}; lr={LR}; batch={BATCH}x{ACCUM}; max_len={MAX_LEN}; steps={global_step}; minutes={train_minutes:.1f}; seed={SEED}"
out_dir = os.path.join(PROJECT, "eval") if not SMOKE else os.path.join(PROJECT, "finetune", "_smoke_eval")
p1, p2 = ft.write_comparison(summary, per_case, manifest, out_dir, BASE_MODEL, os.path.basename(ADAPTER_DIR), info)
print(f"{'scope':<13}{'cases':>6}{'F1 base':>9}{'F1 adapter':>12}{'gain':>8}{'JSON ok base':>14}{'JSON ok adapter':>17}")
for r in summary: print(f"{r['scope']:<13}{r['cases']:>6}{r['f1_base']:>9.3f}{r['f1_adapter']:>12.3f}{r['gain_f1']:>+8.3f}{r['valid_json_base']:>14.2f}{r['valid_json_adapter']:>17.2f}")
print("wrote", p1, "and", p2)
if SMOKE: print("SMOKE TEST: random tiny model. These scores are meaningless and were written to finetune/_smoke_eval, not to eval/.")

In [ ]:
# 10. Save the adapter (a few tens of MB) and, on Colab, download it together with the comparison files.
if not SMOKE:
    model.save_pretrained(ADAPTER_DIR); tok.save_pretrained(ADAPTER_DIR)
    import shutil
    bundle = shutil.make_archive(os.path.join(PROJECT, "adapter_and_comparison"), "zip", PROJECT, "finetune/adapter_qwen25_agri")
    with zipfile.ZipFile(bundle, "a") as z:
        for p in (p1, p2): z.write(p, os.path.relpath(p, PROJECT))
    print("saved", ADAPTER_DIR, "and", bundle)
    try:
        from google.colab import files; files.download(bundle)
    except Exception: pass

## Using the adapter in the pipeline

Unzip `adapter_and_comparison.zip` into the project folder (it puts the adapter in `finetune/adapter_qwen25_agri/` and the two CSV files in `eval/`). Then:

```
pip install torch transformers peft
python run.py --input data/raw/Bhiwani.pdf --db demo.db --llm hf --adapter finetune/adapter_qwen25_agri --view
```

- `--llm hf` loads the base model with transformers (first run downloads about 3 GB); without `--adapter` it uses the base model alone.
- `--policy fill` (default) asks the model only where the rules leave a gap. `--policy prefer` lets a valid model answer override the rules; only switch to it if the adapter beat the rules on your held-out real files.
- On a laptop without a GPU the 1.5B model takes several seconds per call. Ollama (`--llm ollama`, after `ollama pull qwen2.5:1.5b`) is usually faster on CPU, but it runs the base model. To serve the adapter through Ollama you would merge it into the base model and convert the result to GGUF with llama.cpp; that route has not been tested in this project.

## What to write in the report

Copy the table printed by step 9 and state, in plain words: the base model and the adapter, how many held-out cases, whether any were real files, that train and held-out share no file and no prompt (both overlap counts are 0), and what the gain was, including when it is zero or negative.